# Day 013：从 generated_ids 到回答文本

本 Notebook 与 Day 013 互动档案配套，重现 eval_llm.py 在 model.generate() 返回之后的外层数据流。它不重新加载完整模型，而是使用小张量观察完整序列切片、回答解码、对话历史和速度统计。

## 1. 完整序列与新生成部分

MiniMind 原生 generate() 默认返回 prompt token 与新生成 token 拼接后的完整序列。eval_llm.py 根据原始 prompt 长度切片，只把新生成部分交给 decode。

In [ ]:
import torch

prompt_ids = torch.arange(21)
new_ids = torch.arange(100, 125)
generated_ids = torch.cat([prompt_ids, new_ids]).unsqueeze(0)

new_part = generated_ids[0][len(prompt_ids):]

print("prompt shape：", prompt_ids.shape)
print("generated_ids shape：", generated_ids.shape)
print("新生成部分 shape：", new_part.shape)

assert tuple(generated_ids.shape) == (1, 46)
assert tuple(new_part.shape) == (25,)

## 2. gen_tokens 统计 token，不统计字符

源码通过完整序列长度减去 prompt 长度得到生成 token 数。如果 EOS 触发停止，它已经被追加到 generated_ids，因此计入 gen_tokens；decode 时可通过 skip_special_tokens=True 隐藏。

In [ ]:
prompt_len = 21
full_len = 46
gen_tokens = full_len - prompt_len

print("gen_tokens：", gen_tokens)
assert gen_tokens == 25

## 3. TextStreamer 与 response 是两条用途不同的路径

TextStreamer 接收每一步的 next_token，负责边生成边显示给人。生成结束后，程序仍从 generated_ids 切片并 decode，得到完整 response 字符串，供 conversation、文件或接口继续使用。

In [ ]:
streamed_fragments = ["我", "是", "Mini", "Mind", "。"]
response = "".join(streamed_fragments)

print("模拟流式片段：", streamed_fragments)
print("程序保存的完整 response：", response)
assert response == "我是MiniMind。"

## 4. historys 控制保留多少条消息

conversation 中每个 user 或 assistant 字典是一条消息。源码使用 conversation[-args.historys:]，所以 historys=2 会保留上一轮的 user 和 assistant 两条消息，再追加本轮 user。保留的消息会重新应用 chat template 并重新 Tokenizer；当前实现不跨两轮 generate() 保留 KV Cache。

In [ ]:
conversation = [
    {"role": "user", "content": "你是谁"},
    {"role": "assistant", "content": "我是MiniMind。"},
]

historys = 2
conversation = conversation[-historys:] if historys else []
conversation.append({"role": "user", "content": "你能做什么？"})

print(conversation)
assert [m["role"] for m in conversation] == ["user", "assistant", "user"]

默认 historys=0 时，下一轮开头直接得到空列表，上一轮 assistant 回答不会进入下一轮模型输入。

In [ ]:
conversation = [
    {"role": "user", "content": "你是谁"},
    {"role": "assistant", "content": "我是MiniMind。"},
]

historys = 0
conversation = conversation[-historys:] if historys else []

print("下一轮开始时：", conversation)
assert conversation == []

## 5. tokens/s 的真实计时边界

st 在 model.generate() 前记录，最终速度在 decode、conversation.append 和 gen_tokens 之后计算。它主要包含 prompt prefill、逐 token decode、采样和流式显示，所以是近似生成吞吐量，不是隔离测得的纯 decode 吞吐量。

In [ ]:
gen_tokens = 32
elapsed_seconds = 0.5
speed = gen_tokens / elapsed_seconds

print(f"[Speed]: {speed:.2f} tokens/s")
assert speed == 64.0

## 6. 外层对象与类型

    prompt                 str
    conversation           list[dict]
    chat template 结果      str
    tokenizer 结果          BatchEncoding
    input_ids              Tensor [batch, prompt_len]
    generated_ids          Tensor [batch, prompt_len + generated_len]
    新生成切片             Tensor [generated_len]
    response               str
    gen_tokens             int
    speed                  float

input_ids 的内容是 chat template 完整 prompt 对应的 token ID，包括角色与特殊标记，不只是用户问题原文。

## 自测

1. TextStreamer 已经显示回答后，为什么仍需要 decode？
2. historys=2 时，第二轮会重新 Tokenizer 哪三条消息？
3. 跨两轮用户提问是否直接复用上一轮 KV Cache？
4. prompt 长度 21、完整返回长度 46 时，gen_tokens 是多少？
5. 当前 Speed 是纯 decode 速度还是近似生成吞吐量？